## Ajustar um Modelo de Regressão Linear aos dados de Imóveis

**Aluno:** Fabio Kishino

**Objetivo:** Avaliar o modelo usando os dados do arquivo "Imoveis_Fabro_teste.csv" usando a métrica do "Erro quadrático Médio".

In [1]:
# Import
import pandas as pd
import statsmodels.api as sm

from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

In [2]:
def load_data(file_path):
    return pd.read_csv(file_path, sep=';')

df_train = load_data('Imoveis_Fabro_treino.csv')
df_test = load_data('Imoveis_Fabro_teste.csv')

df_test.head()


,Índice,Bairro,Tamanho (m²),N° de Quartos,Tipo,Vagas de Garagem,Unnamed: 6
0,1,Batel,75,2,Apartamento,1,900000
1,2,Batel,105,3,Apartamento,2,1260000
2,3,Batel,150,3,Sobrado,3,1980000
3,4,Batel,120,3,Sobrado,2,1584000
4,5,Batel,240,4,Casa,4,3456000


In [3]:
df_train.drop(columns=['Índice'], inplace=True)
df_test.drop(columns=['Índice'], inplace=True)
df_test.rename(columns={'Unnamed: 6': 'Valor de Venda (R$)'}, inplace=True)


In [4]:
# One Hot Encoding para variáveis categóricas
colunas_categoricas = ['Bairro', 'Tipo']

df_encoded_train = pd.get_dummies(df_train, columns=colunas_categoricas, drop_first=True, dtype=int)
df_encoded_test = pd.get_dummies(df_test, columns=colunas_categoricas, drop_first=True, dtype=int)

df_encoded_train.head()


,Tamanho (m²),N° de Quartos,Vagas de Garagem,Valor de Venda (R$),Bairro_Batel,Bairro_Bigorrilho,Bairro_CIC,Bairro_Cabral,Bairro_Campo Comprido,Bairro_Centro,Bairro_Juvevê,Bairro_Pilarzinho,Bairro_Portão,Bairro_Água Verde,Tipo_Casa,Tipo_Sobrado
0,220,4,4,3168000,1,0,0,0,0,0,0,0,0,0,1,0
1,140,3,2,1680000,1,0,0,0,0,0,0,0,0,0,0,0
2,180,3,3,2376000,1,0,0,0,0,0,0,0,0,0,0,1
3,95,2,1,1140000,1,0,0,0,0,0,0,0,0,0,0,0
4,300,5,4,4320000,1,0,0,0,0,0,0,0,0,0,1,0


In [5]:
df_encoded_train.dtypes

Tamanho (m²)              int64
N° de Quartos             int64
Vagas de Garagem          int64
Valor de Venda (R$)      object
Bairro_Batel              int64
Bairro_Bigorrilho         int64
Bairro_CIC                int64
Bairro_Cabral             int64
Bairro_Campo Comprido     int64
Bairro_Centro             int64
Bairro_Juvevê             int64
Bairro_Pilarzinho         int64
Bairro_Portão             int64
Bairro_Água Verde         int64
Tipo_Casa                 int64
Tipo_Sobrado              int64
dtype: object

In [6]:
coluna_mover = df_encoded_train.pop('Valor de Venda (R$)')
df_encoded_train['Valor de Venda (R$)'] = coluna_mover

coluna_mover = df_encoded_test.pop('Valor de Venda (R$)')
df_encoded_test['Valor de Venda (R$)'] = coluna_mover

df_encoded_test.head()


,Tamanho (m²),N° de Quartos,Vagas de Garagem,Bairro_Batel,Bairro_Bigorrilho,Bairro_CIC,Bairro_Cabral,Bairro_Campo Comprido,Bairro_Centro,Bairro_Juvevê,Bairro_Pilarzinho,Bairro_Portão,Bairro_Água Verde,Tipo_Casa,Tipo_Sobrado,Valor de Venda (R$)
0,75,2,1,1,0,0,0,0,0,0,0,0,0,0,0,900000
1,105,3,2,1,0,0,0,0,0,0,0,0,0,0,0,1260000
2,150,3,3,1,0,0,0,0,0,0,0,0,0,0,1,1980000
3,120,3,2,1,0,0,0,0,0,0,0,0,0,0,1,1584000
4,240,4,4,1,0,0,0,0,0,0,0,0,0,1,0,3456000


In [7]:
def converter_valor_venda(valor):
    """Converte o valor de venda de object para float."""
    if isinstance(valor, str):
        valor = valor.replace('.', '').replace(',', '.')
    return float(valor)

df_encoded_train['Valor de Venda (R$)'] = df_encoded_train['Valor de Venda (R$)'].apply(converter_valor_venda)
df_encoded_test['Valor de Venda (R$)'] = df_encoded_test['Valor de Venda (R$)'].apply(converter_valor_venda)


In [8]:
df_encoded_train.dtypes

Tamanho (m²)               int64
N° de Quartos              int64
Vagas de Garagem           int64
Bairro_Batel               int64
Bairro_Bigorrilho          int64
Bairro_CIC                 int64
Bairro_Cabral              int64
Bairro_Campo Comprido      int64
Bairro_Centro              int64
Bairro_Juvevê              int64
Bairro_Pilarzinho          int64
Bairro_Portão              int64
Bairro_Água Verde          int64
Tipo_Casa                  int64
Tipo_Sobrado               int64
Valor de Venda (R$)      float64
dtype: object

In [9]:
df_encoded_train.head()

,Tamanho (m²),N° de Quartos,Vagas de Garagem,Bairro_Batel,Bairro_Bigorrilho,Bairro_CIC,Bairro_Cabral,Bairro_Campo Comprido,Bairro_Centro,Bairro_Juvevê,Bairro_Pilarzinho,Bairro_Portão,Bairro_Água Verde,Tipo_Casa,Tipo_Sobrado,Valor de Venda (R$)
0,220,4,4,1,0,0,0,0,0,0,0,0,0,1,0,3168000.0
1,140,3,2,1,0,0,0,0,0,0,0,0,0,0,0,1680000.0
2,180,3,3,1,0,0,0,0,0,0,0,0,0,0,1,2376000.0
3,95,2,1,1,0,0,0,0,0,0,0,0,0,0,0,1140000.0
4,300,5,4,1,0,0,0,0,0,0,0,0,0,1,0,4320000.0


In [10]:
# Train data
X_train = df_encoded_train.iloc[:, :-1]
y_train = df_encoded_train['Valor de Venda (R$)']

# Test data
X_test = df_encoded_test.iloc[:, :-1]
y_test = df_encoded_test['Valor de Venda (R$)']

X_intercept = sm.add_constant(X_train)

In [11]:
# Cria um objeto de regressão linear 
regr = LinearRegression()
regr.fit(X_train, y_train)

y_pred = regr.predict(X_test)
print("Predição:", y_pred)

# Coeficientes
print("\nCoeficientes:", regr.coef_)

# Mean squared error (MSE)
print("\nMean squared error: %.2f" % mean_squared_error(y_test, y_pred))

# Root Mean Squared Error (RMSE)
rmse = root_mean_squared_error(y_test, y_pred)

print("\nRoot Mean Squared Error: %.2f" % rmse)
# O coeficiente de determinação (ou R2)
print("\nR2 (ou coeficiente de determinação): %.4f" % r2_score(y_test, y_pred))


Predição: [1154495.91645288 1438403.15624056 2011731.87796112 1639522.25083241
 3097149.65380534 2451067.51519182  865064.18863557  957260.67038388
 1500720.40433175 1843061.04368777 2243797.54081996 3095284.06304709
  587962.51920321  889222.28183651 1168844.23260243 1511184.87195845
 2137160.01217407 2215662.86834814  639639.57072443  896159.55936363
 1165746.78182669 1439619.2933115  2045760.1740573  2486437.92905721
  634588.81064084  986964.17829972 1305185.26916416 1559223.52117915
 1794224.15363737 2352003.90490986  691289.71503591  817014.08182828
  902443.53793063 1244784.17728665 1596886.80601745 2017730.30154754
  354760.31776064  526636.81633019  809917.66436749 1080131.27858093
 1416058.54511328 1727353.03604944  527061.3449558   687725.95457532
  834070.54687021 1244879.31409743 1528513.81495703 1949357.31048712
  276889.36907606  506022.10656678  704659.46453442  939932.83592072
 1360503.59252273 1822427.96477554  222392.78556669  134090.39822566
  677550.13217354  82452